# Deep JSCC Encoder Constellation Visualization

This notebook loads AWGN checkpoints and creates constellation plots of the signal after the encoder step.
The constellation diagram shows the distribution of complex symbols (I-Q components) transmitted through the channel.

## Section 1: Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import OrderedDict
from PIL import Image
from torchvision import transforms
import os
import sys

# Add the project root to path
os.chdir('../')
sys.path.append('./')
print(f"Working directory: {os.getcwd()}")

from model import DeepJSCC
from utils import image_normalization

# Configure matplotlib for Jupyter
plt.rcParams['figure.figsize'] = (12, 5)
print("Libraries imported successfully")

## Section 2: Load Checkpoint and Extract Parameters

In [ ]:
def load_checkpoint(checkpoint_path, device='cpu'):
    """Load model from checkpoint, handling DataParallel wrapper"""
    state_dict = torch.load(checkpoint_path, map_location=device)
    
    # Remove 'module.' prefix if model was saved with DataParallel
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace('module.', '')
        new_state_dict[name] = v
    
    return new_state_dict

def get_checkpoint_params(checkpoint_dir):
    """Extract parameters from checkpoint directory name"""
    # Format: CIFAR10_c_snr_ratio_channel_timestamp
    parts = Path(checkpoint_dir).name.split('_')
    try:
        c = int(parts[1])
        snr = float(parts[2])
        ratio = float(parts[3])
        channel = parts[4]
        return c, snr, ratio, channel
    except (IndexError, ValueError) as e:
        raise ValueError(f"Could not parse checkpoint directory: {checkpoint_dir}") from e

def find_awgn_checkpoints(checkpoint_dir):
    """Find all AWGN checkpoints in directory"""
    checkpoint_dir = Path(checkpoint_dir)
    awgn_dirs = [d for d in checkpoint_dir.iterdir() if d.is_dir() and 'AWGN' in d.name]
    return sorted(awgn_dirs)

print("Checkpoint utility functions defined")

## Section 3: Initialize Deep JSCC Model

Select and load a checkpoint from the available AWGN models.

In [ ]:
# Find available AWGN checkpoints
checkpoint_base_dir = './out/checkpoints'
awgn_checkpoints = find_awgn_checkpoints(checkpoint_base_dir)

print(f"Found {len(awgn_checkpoints)} AWGN checkpoints:")
for i, ckpt in enumerate(awgn_checkpoints[:5]):
    print(f"  {i}: {ckpt.name}")
if len(awgn_checkpoints) > 5:
    print(f"  ... and {len(awgn_checkpoints) - 5} more")

# Select the first AWGN checkpoint
selected_ckpt_dir = awgn_checkpoints[0]
print(f"\nUsing checkpoint: {selected_ckpt_dir.name}")

# Extract parameters
c, snr_ckpt, ratio, channel = get_checkpoint_params(selected_ckpt_dir)
print(f"Checkpoint parameters: c={c}, SNR={snr_ckpt}, Ratio={ratio}, Channel={channel}")

# Find epoch file
epoch_files = sorted(selected_ckpt_dir.glob('epoch_*.pkl'))
if epoch_files:
    checkpoint_path = epoch_files[-1]  # Use latest epoch
    print(f"Using checkpoint file: {checkpoint_path.name}")
else:
    raise FileNotFoundError(f"No checkpoint files found in {selected_ckpt_dir}")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model
model = DeepJSCC(c=c, channel_type=channel, snr=snr_ckpt)
state_dict = load_checkpoint(str(checkpoint_path), device=device)
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

print("Model loaded successfully")

## Section 4: Register Encoder Hook to Capture Output

In [ ]:
class EncoderHook:
    """Hook to capture encoder output"""
    def __init__(self):
        self.features = None
    
    def __call__(self, module, input, output):
        self.features = output.detach()

# Register hook
encoder_hook = EncoderHook()
model.encoder.register_forward_hook(encoder_hook)
print("Encoder hook registered")

## Section 5: Prepare Test Image or Generate Random Input

In [ ]:
# Try to load a test image, fall back to random tensor
test_image_path = './demo/kodim08.png'

if os.path.exists(test_image_path):
    print(f"Loading test image: {test_image_path}")
    image = Image.open(test_image_path)
    transform = transforms.Compose([transforms.ToTensor()])
    x = transform(image).to(device)
    # Add batch dimension
    x = x.unsqueeze(0)
    print(f"Test image shape: {x.shape}")
else:
    print("Test image not found, generating random input tensor")
    x = torch.randn(1, 3, 256, 256).to(device)
    print(f"Random tensor shape: {x.shape}")

# Optional: normalize the image
# x = image_normalization('normalization')(x)

## Section 6: Forward Pass Through Encoder

In [ ]:
# Execute forward pass through encoder only (without channel transmission)
with torch.no_grad():
    encoder_output = model.encoder(x)

# The encoder hook will capture the output
encoder_output_from_hook = encoder_hook.features

print(f"Encoder output shape: {encoder_output.shape}")
print(f"Encoder output dtype: {encoder_output.dtype}")
print(f"Encoder output range: [{encoder_output.min():.4f}, {encoder_output.max():.4f}]")
print(f"Encoder output mean: {encoder_output.mean():.4f}, std: {encoder_output.std():.4f}")

## Section 7: Reshape Encoder Output to Symbol Format

Convert the encoder output to a format suitable for constellation plotting: (num_symbols, 2*c)

In [ ]:
# Reshape encoder output to (num_symbols, 2*c) format
if encoder_output.dim() == 4:
    # Shape: (batch, channels, height, width) -> (num_symbols, channels)
    batch, channels, height, width = encoder_output.shape
    num_symbols = batch * height * width
    symbols = encoder_output.permute(0, 2, 3, 1).reshape(num_symbols, channels)
elif encoder_output.dim() == 3:
    # Shape: (channels, height, width) -> (num_symbols, channels)
    channels, height, width = encoder_output.shape
    num_symbols = height * width
    symbols = encoder_output.permute(1, 2, 0).reshape(num_symbols, channels)
else:
    symbols = encoder_output.reshape(-1, encoder_output.shape[-1])
    num_symbols = symbols.shape[0]

print(f"Reshaped symbols: {symbols.shape}")
print(f"Number of symbols: {num_symbols}")
print(f"Number of channels (2*c): {channels}")
print(f"Number of complex symbols (c): {channels // 2}")

# Move to CPU for visualization
symbols_cpu = symbols.cpu().numpy()

## Section 8: Create Constellation Plots for Each Symbol

Create I-Q (in-phase/quadrature) scatter plots for each complex symbol.

In [ ]:
# Create constellation plots
num_symbols_to_plot = channels // 2

# Create output directory
constellation_dir = Path('../constellations')
constellation_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, num_symbols_to_plot, figsize=(6*num_symbols_to_plot, 5))

# Handle single subplot case
if num_symbols_to_plot == 1:
    axes = [axes]

for i in range(num_symbols_to_plot):
    # Extract I and Q components
    # First c components are I (in-phase), next c are Q (quadrature)
    I = symbols_cpu[:, i]
    Q = symbols_cpu[:, i + num_symbols_to_plot]
    
    # Create scatter plot
    axes[i].scatter(I, Q, alpha=0.5, s=20, edgecolors='none')
    axes[i].set_xlabel('I (In-phase)', fontsize=11)
    axes[i].set_ylabel('Q (Quadrature)', fontsize=11)
    axes[i].set_title(f'Symbol {i+1} Constellation', fontsize=12, fontweight='bold')
    axes[i].grid(True, alpha=0.3)
    axes[i].axhline(y=0, color='k', linewidth=0.5, alpha=0.5)
    axes[i].axvline(x=0, color='k', linewidth=0.5, alpha=0.5)
    
    # Set equal aspect ratio
    axes[i].set_aspect('equal', adjustable='box')
    
    # Add statistics
    mean_power = np.sqrt(I**2 + Q**2).mean()
    axes[i].text(0.02, 0.98, f'Power: {mean_power:.3f}', 
                transform=axes[i].transAxes, fontsize=10,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Extract parameters from checkpoint directory name for proper naming
parts = selected_ckpt_dir.name.split('_')
c_val = int(parts[1])
snr_val = float(parts[2])
ratio_val = float(parts[3])
channel_val = parts[4]

# Create filename with proper naming convention
filename = f"constellation_c{c_val}_snr{snr_val:.1f}dB_ratio{ratio_val:.2f}_{channel_val}.png"
save_path = constellation_dir / filename

fig.suptitle(f'Encoder Constellation - c={c_val}, SNR={snr_val}dB, Ratio={ratio_val}, Channel={channel_val}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()

# Save figure
plt.savefig(str(save_path), dpi=150, bbox_inches='tight')
print(f"Constellation plot saved to: {save_path}")
plt.show()

## Section 9: Visualize Multiple AWGN Checkpoints

Compare constellation patterns across different checkpoints (different SNR values and training epochs)

In [ ]:
# Function to get constellation for a checkpoint
def get_constellation_from_checkpoint(ckpt_path, model_c, model_channel, device, input_tensor):
    """Get constellation symbols from a checkpoint"""
    state_dict = load_checkpoint(str(ckpt_path), device=device)
    temp_model = DeepJSCC(c=model_c, channel_type=model_channel, snr=2000)
    temp_model.load_state_dict(state_dict)
    temp_model = temp_model.to(device)
    temp_model.eval()
    
    with torch.no_grad():
        output = temp_model.encoder(input_tensor)
    
    return output

# Compare first 3 AWGN checkpoints
num_to_compare = min(3, len(awgn_checkpoints))
comparison_results = []

for idx in range(num_to_compare):
    ckpt_dir = awgn_checkpoints[idx]
    c_val, snr_val, ratio_val, channel_val = get_checkpoint_params(ckpt_dir)
    
    epoch_files = sorted(ckpt_dir.glob('epoch_*.pkl'))
    if not epoch_files:
        print(f"Skipping {ckpt_dir.name} - no checkpoint files")
        continue
    
    ckpt_file = epoch_files[-1]
    
    try:
        output = get_constellation_from_checkpoint(ckpt_file, c_val, channel_val, device, x)
        
        # Reshape for visualization
        if output.dim() == 4:
            batch, chan, h, w = output.shape
            reshaped = output.permute(0, 2, 3, 1).reshape(batch*h*w, chan)
        elif output.dim() == 3:
            chan, h, w = output.shape
            reshaped = output.permute(1, 2, 0).reshape(h*w, chan)
        else:
            reshaped = output.reshape(-1, output.shape[-1])
        
        comparison_results.append({
            'name': ckpt_dir.name,
            'snr': snr_val,
            'symbols': reshaped.cpu().numpy()
        })
        print(f"✓ Loaded {ckpt_dir.name}")
    except Exception as e:
        print(f"✗ Error loading {ckpt_dir.name}: {e}")

print(f"\nLoaded {len(comparison_results)} checkpoints for comparison")

In [ ]:
# Plot comparison of multiple checkpoints
if comparison_results:
    constellation_dir = Path('../constellations')
    constellation_dir.mkdir(exist_ok=True)
    
    # Plot first symbol across multiple checkpoints
    fig, axes = plt.subplots(1, len(comparison_results), figsize=(6*len(comparison_results), 5))
    
    if len(comparison_results) == 1:
        axes = [axes]
    
    symbol_idx = 0  # Plot only first symbol
    snr_values = []
    
    for ax_idx, result in enumerate(comparison_results):
        symbols_cpu = result['symbols']
        c_val = symbols_cpu.shape[1] // 2
        
        I = symbols_cpu[:, symbol_idx]
        Q = symbols_cpu[:, symbol_idx + c_val]
        
        axes[ax_idx].scatter(I, Q, alpha=0.5, s=20, edgecolors='none')
        axes[ax_idx].set_xlabel('I (In-phase)', fontsize=11)
        axes[ax_idx].set_ylabel('Q (Quadrature)', fontsize=11)
        axes[ax_idx].set_title(f'SNR={result["snr"]} dB', fontsize=12, fontweight='bold')
        axes[ax_idx].grid(True, alpha=0.3)
        axes[ax_idx].axhline(y=0, color='k', linewidth=0.5, alpha=0.5)
        axes[ax_idx].axvline(x=0, color='k', linewidth=0.5, alpha=0.5)
        axes[ax_idx].set_aspect('equal', adjustable='box')
        snr_values.append(result["snr"])
    
    # Create comparison filename with SNR range
    min_snr = min(snr_values)
    max_snr = max(snr_values)
    comparison_filename = f"constellation_comparison_snr{min_snr:.1f}-{max_snr:.1f}dB_symbol{symbol_idx+1}.png"
    comparison_path = constellation_dir / comparison_filename
    
    fig.suptitle(f'Symbol {symbol_idx+1} Constellation Comparison - SNR Range: {min_snr}dB to {max_snr}dB', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save figure
    plt.savefig(str(comparison_path), dpi=150, bbox_inches='tight')
    print(f"Comparison plot saved to: {comparison_path}")
    plt.show()
else:
    print("No comparison data available")